# Ticket Triage: train your first LoRA adapter

This notebook teaches the original **Qwen/Qwen3-0.6B**, commit `c1899de289a04d12100db370d81485cdf75e47ca`, to produce ticket JSON. It trains on 128 training examples, checks 34 validation examples, and does not evaluate test tickets.

Your original validation baseline is recorded in `training/baseline_reference.json`. Training settings below are a starting experiment, not settings already proven best. We select the lowest validation-loss checkpoint, then measure its actual predictions. Lower loss does not guarantee better task scores.

**Before starting:** preserve your downloaded baseline results. Open this notebook in a fresh Colab GPU runtime. Upload the newly regenerated `artifacts/colab-project.zip` when asked. Your previous ZIP does not include the training code. No Hugging Face token is needed to download the public base; this notebook does not publish anything.

Run the cells in order. GPU training has not yet been tested for this project; the smoke step checks compatibility before the full pilot. If a cell fails, stop and share its error text.


## 1. Connect and check the GPU

Select Runtime → Change runtime type → GPU, then connect. Colab supplies the GPU; your laptop is not training.

In [ ]:
import sys
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "No GPU connected. Select a GPU runtime, reconnect, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 2. Upload the NEW project ZIP

Choose the updated `artifacts/colab-project.zip`. A fresh runtime avoids mixing previous experiment files.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io
import os
import shutil

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one colab-project.zip file."
zip_bytes = next(iter(uploaded.values()))
project = Path("/content/ticket-triage")
assert not project.exists(), "Project already exists. Skip this cell if already uploaded, or use a fresh runtime."
staging = Path("/content/triage-upload")
staging.mkdir(exist_ok=True)
with ZipFile(io.BytesIO(zip_bytes)) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        if relative.is_absolute() or ".." in relative.parts or not relative.parts or relative.parts[0] != "ticket-triage":
            raise ValueError("Unexpected path in ZIP")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Symbolic links are not accepted")
        if member.is_dir():
            continue
        target = staging / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
(staging / "ticket-triage").rename(project)
os.chdir(project)
print("Working folder:", project)
del uploaded, zip_bytes


## 3. Install training libraries and check files

Transformers stays at the baseline version. PEFT adds LoRA and Accelerate supports training. Gradio/Diffusers are not used. If Colab asks for a session restart, restart then continue here without uploading again. Subprocess output is explicitly printed so errors are visible.

In [ ]:
import os
from pathlib import Path
os.chdir("/content/ticket-triage")
%pip install -q -r training/requirements-training.txt
import subprocess
import sys
import json

def run(*arguments):
    # Stream output line by line, including training logs and Python errors.
    process = subprocess.Popen([sys.executable, "-u", *map(str, arguments)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Command failed. Read the output above and stop here.")

ROOT = Path("/content/ticket-triage")
reference = json.loads((ROOT / "training/baseline_reference.json").read_text())
print("Model:", reference["model_id"])
print("Fixed revision:", reference["resolved_revision"])
print("Recorded original scores:", reference["metrics"])
run("-m", "evaluation.validate_data")


## 4. Save runs on Google Drive

Colab storage disappears when the runtime is deleted. Mount your Drive so epoch checkpoints, optimizer state, and exported adapters survive. Authorize your own Drive in Colab when asked. No token goes into project code. The notebook uses a new folder per experiment and does not overwrite your original baseline.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount("/content/drive")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT = Path("/content/drive/MyDrive/ticket-triage-runs") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print("Persistent run folder:", RUN_ROOT)
print("Keep this path if you need to reconnect later.")


## 5. Inspect what the model will learn

This downloads only the tokenizer. For every example, the training script builds:

```text
system instructions + ticket → masked (not scored during learning)
correct JSON answer + end-of-turn token → learned
```

The output prints `supervised_answer`. It should contain only the expected JSON and an end-of-turn marker, never the ticket or instructions. The code refuses to truncate an example beyond 512 tokens. The checks also verify that the prompt and validation file match your baseline.


In [ ]:
run("-m", "training.train_lora", "--output-dir", RUN_ROOT / "label-check", "--check-only")


## 6. Run two training steps first

This trains a disposable smoke adapter on eight training examples and checks eight validation examples. It tests GPU memory, loss calculation, checkpoint saving, and adapter export. It is not the final trained model. Expect finite training/validation losses; if it fails or losses are not finite, stop here.

Training uses float32 parameters with FP16 mixed precision and gradient checkpointing. This avoids assuming that T4 has native BF16 training support. Only LoRA parameters are updated.


In [ ]:
run("-m", "training.train_lora", "--output-dir", RUN_ROOT / "smoke", "--smoke")
smoke = json.loads((RUN_ROOT / "smoke/training_metadata.json").read_text())
assert smoke["status"] == "complete"
print("Smoke test passed. Best validation loss:", smoke["best_validation_loss"])


## 7. Train the real pilot adapter

Start with 3 epochs, learning rate 0.0002, LoRA rank 16, batch size 1, and 8 accumulated batches per optimizer step. One epoch means one pass over the 128 training examples. There are about 16 optimizer steps per epoch, 48 total.

After each epoch, the script measures answer-token loss on validation examples and saves a checkpoint on Drive. It keeps the best checkpoint plus recent checkpoint(s) within the save limit, then exports the selected adapter to `training/adapter` inside this run. Validation answers calculate loss; they are not used for gradient updates. The test set is not used for model selection.

If disconnected after a checkpoint, remount Drive and use the same run path with `--resume-from-checkpoint /path/to/checkpoint-N`. Model/data/settings must match. Work after the last saved epoch may need to be repeated. Do not rerun a completed run into the same output folder.


In [ ]:
TRAIN_DIR = RUN_ROOT / "training"
run("-m", "training.train_lora", "--output-dir", TRAIN_DIR, "--epochs", "3")
trained = json.loads((TRAIN_DIR / "training_metadata.json").read_text())
assert trained["status"] == "complete"
print("Selected checkpoint:", trained["best_checkpoint"])
print("Best validation loss:", trained["best_validation_loss"])
print("Adapter:", TRAIN_DIR / "adapter")


## 8. Evaluate original and adapted models under the same current environment

Adding training libraries changes the runtime environment. Rerun the original model, then evaluate the selected adapter using the same code, fixed model revision, prompt, validation tickets, and greedy generation settings. This preserves your earlier baseline while providing a matched comparison now. Each model loads in a separate process to free GPU memory between runs.

The model sees only instructions and ticket text during generation. Expected answers are compared afterward by the evaluator. Adapter identity checks prevent loading it onto an incompatible original model. Test data remains unused.


In [ ]:
for name, adapter in (("base-validation", None), ("adapted-validation", TRAIN_DIR / "adapter")):
    destination = RUN_ROOT / name
    arguments = ["-m", "evaluation.run_model", "--input", "data/validation.jsonl",
                 "--model", reference["model_id"], "--revision", reference["resolved_revision"],
                 "--device", "cuda", "--output-dir", destination]
    if adapter is not None:
        arguments += ["--adapter", adapter]
    run(*arguments)
    run("-m", "evaluation.evaluate", "--gold", "data/validation.jsonl",
        "--predictions", destination / "predictions.jsonl", "--output", destination / "metrics.json")


## 9. Compare scores and review summaries

Higher values are better for these four metrics. Schema validity checks format, macro-F1 averages category F1 equally, and environment/team accuracy count correct usable outputs. Invalid-schema responses count as wrong across task metrics. Team follows category; summary factuality is not automatically scored.

The saved adapter was selected by validation loss, not by test performance. Compare task scores before claiming improvement. Even if scores improve on this small AI-authored dataset, real-world quality is not established.


In [ ]:
base = json.loads((RUN_ROOT / "base-validation/metrics.json").read_text())
adapted = json.loads((RUN_ROOT / "adapted-validation/metrics.json").read_text())
print(f"{'metric':27} {'first base':>11} {'base now':>11} {'adapted':>11} {'change':>11}")
for key in reference["metrics"]:
    print(f"{key:27} {reference['metrics'][key]:11.3f} {base[key]:11.3f} {adapted[key]:11.3f} {adapted[key]-base[key]:+11.3f}")
from evaluation.validate_data import read_jsonl
predictions = {r["id"]: r for r in read_jsonl(RUN_ROOT / "adapted-validation/predictions.jsonl")}
for row in read_jsonl(ROOT / "data/validation.jsonl"):
    print("\n", row["id"], row["ticket"])
    print("Expected:", json.dumps(row["expected"]))
    print("Adapted:", predictions[row["id"]]["prediction"])
print("Review whether summaries preserve facts and avoid invented claims; wording need not match exactly.")


## 10. Download an adapter and results backup

Checkpoints remain on Drive for resume. Download a smaller ZIP with the selected adapter, tokenizer, logs, metadata, and original/adapted validation results (excluding optimizer checkpoints and the original model weights). The adapter alone is not a complete standalone model: it needs the recorded base model revision.

Do not publish or evaluate test data yet. First inspect the validation comparison and decide whether training settings should change. Keep every experiment's settings and results rather than overwriting them.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import importlib.metadata
packages = sorted(f"{d.metadata['Name']}=={d.version}" for d in importlib.metadata.distributions() if d.metadata.get("Name"))
(RUN_ROOT / "packages.txt").write_text("\n".join(packages) + "\n")
archive_path = Path("/content") / (RUN_ID + "-lora-results.zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob("*"):
        if path.is_file() and not any(part.startswith("checkpoint-") for part in path.relative_to(RUN_ROOT).parts):
            archive.write(path, path.relative_to(RUN_ROOT))
files.download(str(archive_path))
print("Backup ready. Checkpoints remain on Drive at:", RUN_ROOT)


References: [Transformers Trainer 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer), [PEFT LoRA](https://huggingface.co/docs/peft/quicktour), [Qwen model card](https://huggingface.co/Qwen/Qwen3-0.6B).

Verification status: offline code/data checks run locally; real CUDA training, checkpoint resume, and adapter generation require your Colab smoke run. No improvement or training duration is promised.
